# Does CCNN Structure Augment Attention and Improve Weight-Space Repair?
**A literature-grounded, falsification-first Colab research notebook (experimental protocol and executable exploratory pilot)**

**Research identity:** RQ1-M + RQ6-M • **Status:** implementation/proposal, *not* a validated empirical claim • **Notebook version:** 1.0 • **15 November 2026**

## Abstract

We investigate a single connected hypothesis: whether the **original time-varying, causal CCNN lag-memory structure** contributes useful information to attention and whether its learned kernel representation subsequently permits more effective, localized, non-regressive weight edits than unstructured filters and conventional optimization. The study adapts **multi-query associative recall** from *Zoology* (ICLR 2024) and context-length/extrapolation ideas from *Samba* (ICLR 2025); compares **attention-only, attention + CCNN, attention + expressive MLP-conditioned FIR** in a shared experimental setting; and uses edit-efficacy, locality/preservation and sham editing controls motivated by **AlphaEdit** (ICLR 2025), **CrispEdit** (ICML 2026), and **The Mirage of Model Editing** (ACL 2025). We specify a blinded model-weight fault and original-label-preserving repair intervention. **This is a small adapted diagnostic, not a reproduction of the original papers' benchmark scale or a demonstration of CCNN superiority.** Predictions, numerical editing outcomes, and claims must be generated by execution rather than invented.

## 1. Research question and falsifiable hypotheses

> **Does CCNN's structured lag-memory mechanism enhance an attention-based model's use of temporal evidence, and does that same structure permit more effective, less disruptive weight-space repair?**

**H1 — Attention augmentation:** For an independently defined temporal key–value retrieval problem, attention + CCNN improves held-out retrieval performance and/or relevant-evidence selection **relative to a strong attention + unrestricted conditional FIR** at comparable compute. Attention-only is a reference; an ablation of CCNN's coupling must explain any gain.

**H2 — Repair:** After *parameter* corruption (no label change), a blinded structured edit achieves greater fault-specific recovery with acceptable preservation than no edit, a matched numerical edit, an unstructured edit and ordinary continued optimization. An improvement also seen in a zero-fault placebo is **not** evidence of fault-specific repair.

**Partial outcomes:** H1 and H2 can be supported, rejected, or remain inconclusive separately. Both must be supported to establish the combined claim.

**Important architectural distinction:** The tested lag/PDE–ODE memory **modulates attention logits**. This is a *proposed coupling derived for this study*, not a coupling demonstrated in the original CCNN paper. The response curves and direct change in attention weights can therefore be measured; this differs from V3's independent output-residual branch.

## 2. Source basis and proposed method

### 2.1 Primary CCNN mechanism

Habiba & Pearlmutter (2021) model a causal time-varying lag convolution

\[
m_\theta(t)=\int_0^T K_\theta(\tau,t)\,x(t-\tau)\,d\tau,
\quad \partial_\tau u(t,\tau)=K_\theta(\tau,t)x(t-\tau),\quad
\dot z(t)=f(u(t,T)).
\]

We implement a **discrete, finite-history numerical approximation** to this lag accumulation and a leaky global latent state. This is *not* a claim about continuous solver superiority or spatial diffusion. We retain interpretable delay center, condition-dependent center, width, frequency and chirp.

### 2.2 Proposed attention coupling

For query token \(q\), attention's ordinary compatibility scores \(s_{q,t}\) receive a learned temporal-memory contribution:

\[
s'_{q,t}=s_{q,t}+\gamma z_t,\qquad
\alpha_{q,t}=\operatorname{softmax}_t(s'_{q,t}).
\]

The comparator replaces the structured kernel with **unrestricted MLP-conditioned FIR weights**, while retaining the same attention/logit-coupling interface and latent-state readout. Ablating \(\gamma z_t\) identifies its incremental effect; the test does *not* assume that changing attention weights necessarily improves predictions.

### 2.3 Evidence from recent work and how it maps

| Published method | Paper's experiment | Adaptation here |
|---|---|---|
| **Zoology** (ICLR 2024) | Multi-query associative recall (MQAR), comparing attention/gated convolution and hybrids | Adapted discrete key–value lookup with distractor keys; report correct-key recall and evidence attention |
| **Samba** (ICLR 2025) | Passkey/phonebook retrieval beyond training context | Hold out relevant-event positions/time lags; examine retrieval extrapolation |
| **Hymba** (ICLR 2025) | Complementary attention and state-space components and ablations | Shared attention specification, memory-module comparison, memory-off ablations |
| **AlphaEdit** (ICLR 2025) | Edit efficacy and preservation with null-space constraints | Measure unaffected behavior after repair, not edit loss alone |
| **CrispEdit** (ICML 2026) | Preservation-constrained editing, including nonconverged models | Compare numerical baselines and constrained retention; do not equate undertraining alone with invalidity |
| **Mirage of Model Editing** (ACL 2025) | Realistic editing evaluation vs misleading synthetic/teacher-forced tests | No oracle fault knowledge, no test-time label feedback, zero-fault placebo, separate held-out test |

**Scope limits:** These papers do **not** evaluate this particular CCNN proposal. Our MQAR-style task is a constrained diagnostic, not the official Zoology benchmark. It can falsify a proposed advantage in this setting; it cannot establish general superiority.


### Published MQAR benchmark fidelity (mandatory before publication)

The official Zoology benchmark is not an arbitrary sequence plot. Its [MQAR generator](https://github.com/HazyResearch/zoology/blob/main/zoology/data/multiquery_ar.py) creates **key–value token pairs followed by multiple key queries**, with target value-token labels and ignored non-query positions. It explicitly controls vocabulary size, number of pairs, input length, gap distribution (\`power_a\`), and distractor/non-query values. The authors' [original MQAR sweep configuration](https://github.com/HazyResearch/zoology/blob/main/zoology/experiments/mqar_example_configs/original_mqar_configs.py) uses an 8,192-token vocabulary and multiple input lengths, including held-out longer sequences with many key–value pairs.

**Important difference:** This notebook is a deliberately smaller **binary cue–key association diagnostic** with three events and held-out event positions. It **does not reproduce the official Zoology generator, token-vocabulary classification head, or full MQAR sweep**. The diagnostic is useful for testing the CCNN/attention coupling mechanics but **cannot substitute for official MQAR evaluation** in a paper. Before a confirmatory H1 claim, connect the same architecture variants to the official \`MQARConfig\` input/label/mask contract, use the published model/data-sweep logic and independently audit the pair-count/length extrapolation results. Do not cite pilot figures as published MQAR performance.

## 3. Experimental design and pre-analysis decisions

The **same task** trains three alternatives: **A0 attention only**, **A1 attention + CCNN**, and **A2 attention + unrestricted MLP-FIR**. Every sample contains several cue-marked historical key–value events plus a final query; the target is the sign of the value associated with the queried key. The evaluator records the relevant event position (never an input label). Test queries refer to held-out temporal positions.

**Metrics:** Retrieval accuracy (primary task); correct-evidence top-1 rate and attention mass (mechanistic diagnostics, not sufficient alone); branch-ablation delta in held-out loss; trainable parameters, training time and BCE. Both learned memory methods use the *same downstream leaky state and attention logit interface*. No learned model generates the labels.

**Repair intervention:** Select a saved clean checkpoint, perturb an *actual model-weight parameter*, calibrate **functional loss damage on a calibration set**, and keep the labels fixed. Repair algorithms receive the damaged model and training/validation examples, **not** the fault identity or the clean checkpoint. For CCNN compare structured-5 vs first-order tangent-5 vs unrestricted spectral-5 / full-14 edit and full-network continued Adam. For FIR compare spectral-5 / full-14 and continued Adam. An identically budgeted **no-fault placebo** starts from the clean checkpoint. Report recovery, source retention, placebo improvement, edit parameter count, and compute.

**Selection boundary:** training fits candidates, validation accepts/rejects without inspecting the test set; test is evaluated once for audit. This notebook does not enforce secure process isolation between editor and evaluator—blindness here is a code convention.

**Decision rule:** \`FAST_MODE=True\` is **engineering-only**. With \`FAST_MODE=False\` the notebook conducts a larger **exploratory pilot**. It never labels results confirmatory: a confirmatory study additionally needs a frozen protocol, tuned resource-matched alternative(s), >=10 paired seeds, independent replication, and robust controls including an SSM/generic Neural ODE where the claim requires them.

In [ ]:
# 4. Environment and immutable experimental settings
from __future__ import annotations
import copy, hashlib, json, math, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F

FAST_MODE = True             # engineering smoke by default; False = exploratory pilot
CONFIRMATORY = False          # cannot be overridden inside this notebook
assert not CONFIRMATORY, "Freeze protocol and use an independently audited confirmatory runner"
SEEDS = [11] if FAST_MODE else [11,23,47]
EPOCHS = 2 if FAST_MODE else 45
PATIENCE = 2 if FAST_MODE else 9
REPAIR_STEPS = 3 if FAST_MODE else 50
TRAIN_N = 192 if FAST_MODE else 1500
VAL_N = 96 if FAST_MODE else 400
CAL_N = 96 if FAST_MODE else 400
TEST_N = 128 if FAST_MODE else 550
SEQ, LAGS, WIDTH = 48, 14, 32
OUTPUT = Path("ccnn_attention_repair_paper_outputs")
OUTPUT.mkdir(exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4, torch.get_num_threads()))
print({"device":str(DEVICE), "torch":torch.__version__,
       "mode":"engineering_smoke" if FAST_MODE else "exploratory_pilot", "seeds":SEEDS})

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(s)
def sha_tensors(state):
    h=hashlib.sha256()
    for k,v in sorted(state.items()):
        h.update(k.encode());h.update(v.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()
def get_logits(model, data, **kwargs):
    return model(data["x"],data["q"],**kwargs)[0]
def bce(logit,y):
    return F.binary_cross_entropy_with_logits(logit,y)


In [ ]:
# 5. MQAR-style delayed key–value retrieval, with position extrapolation
# Inspired by the *task class* in Zoology (not their exact data generator).
KEYS=torch.tensor([-.9,.35,.9])
def make_task(seed, split, size):
    offsets={"train":100,"val":200,"cal":250,"test":300}
    assert split in offsets
    g=torch.Generator().manual_seed(seed*1009+offsets[split])
    x=.08*torch.randn(size,SEQ,generator=g)
    q=.01*torch.randn(size,SEQ,generator=g)
    which=torch.randint(3,(size,),generator=g)
    y=torch.randint(2,(size,),generator=g).float()
    signs=torch.where(y>.5,1.,-1.)
    train_positions=torch.arange(10,33)
    unseen_positions=torch.cat((torch.arange(5,10),torch.arange(34,43)))
    pool=unseen_positions if split=="test" else train_positions
    selected=pool[torch.randint(len(pool),(size,),generator=g)]
    # Distractor events: different keys with independently random values.
    all_positions=torch.arange(5,43)
    valid_positions=(all_positions[None,:]!=selected[:,None])
    noise_scores=torch.rand(size,len(all_positions),generator=g)
    noise_scores[~valid_positions]=-1
    others=all_positions[noise_scores.topk(2,dim=1).indices]
    batch=torch.arange(size)
    x[batch,selected]=signs*(.9+.08*torch.randn(size,generator=g))
    q[batch,selected]=KEYS[which]
    other_codes=(which[:,None]+torch.tensor([1,2]))%3
    q[batch[:,None],others]=KEYS[other_codes]
    rand_sign=torch.where(torch.rand(size,2,generator=g)>.5,1.,-1.)
    x[batch[:,None],others]=rand_sign*(.9+.08*torch.randn(size,2,generator=g))
    q[:,-1]=KEYS[which]
    x[:,-1]=0
    return {"x":x.to(DEVICE),"q":q.to(DEVICE),"y":y.to(DEVICE),
            "evidence":selected.to(DEVICE),
            "ids":tuple(f"{seed}:{split}:{i}" for i in range(size)),
            "split":split}
def make_datasets(seed):
    d={name:make_task(seed,name,size) for name,size in
       [("train",TRAIN_N),("val",VAL_N),("cal",CAL_N),("test",TEST_N)]}
    allids=[s for name in d for s in d[name]["ids"]]
    assert len(allids)==len(set(allids))
    assert not set(d["train"]["evidence"].unique().tolist()).intersection(
        set(d["test"]["evidence"].unique().tolist()))
    for split,obj in d.items():
        i=torch.arange(len(obj["y"]),device=DEVICE)
        assert torch.allclose(obj["q"][i,obj["evidence"]],obj["q"][:,-1])
    return d


In [ ]:
# 6. Shared attention and original lag-kernel memory vs unrestricted FIR
def lag_history(x,length=LAGS):
    return torch.stack([F.pad(x[:,:x.shape[1]-j],(j,0)) for j in range(length)],-1)
def leaky_state_trace(m):
    alpha=1-math.exp(-8.0/m.shape[1])
    z=torch.zeros_like(m[:,0]);history=[]
    for t in range(m.shape[1]):
        z=(1-alpha)*z+alpha*torch.tanh(2*m[:,t])
        history.append(z)
    return torch.stack(history,1)

class CCNNMemory(nn.Module):
    def __init__(self):
        super().__init__()
        for name,val in [("mu0",.43),("muq",.12),("logsigma",-2.1),
                         ("freq",2.1),("chirp",.5)]:
            setattr(self,name,nn.Parameter(torch.tensor(val)))
    def params(self):
        return (self.mu0,self.muq,self.logsigma,self.freq,self.chirp)
    def weights(self,q,delta=None):
        p=self.params()
        if delta is not None:
            p=tuple(v+delta[i] for i,v in enumerate(p))
        tau=torch.arange(LAGS,device=q.device,dtype=q.dtype).view(1,1,LAGS)/(LAGS-1)
        mu=p[0]+p[1]*q.unsqueeze(-1)
        width=F.softplus(p[2])+.04
        raw=torch.exp(-.5*((tau-mu)/width).square())*torch.cos(
            2*torch.pi*(p[3]*tau+p[4]*tau.square()))
        return F.normalize(raw,dim=-1,eps=1e-7)
    def trace(self,x,q,k_override=None):
        k=self.weights(q) if k_override is None else k_override
        return leaky_state_trace((lag_history(x)*k).sum(-1))

class FIRMemory(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(5,24),nn.GELU(),nn.Linear(24,LAGS))
    def raw(self,q):
        t=torch.arange(q.shape[1],device=q.device,dtype=q.dtype)[None,:]/q.shape[1]
        t=t.expand_as(q)
        return self.net(torch.stack((q,q.square(),torch.sin(torch.pi*q),t,t.square()),-1))
    def weights(self,q):
        return F.normalize(self.raw(q),dim=-1,eps=1e-7)
    def trace(self,x,q,k_override=None):
        k=self.weights(q) if k_override is None else k_override
        return leaky_state_trace((lag_history(x)*k).sum(-1))

class AttentionMemory(nn.Module):
    def __init__(self,kind="attention"):
        super().__init__()
        assert kind in ("attention","ccnn","fir")
        self.kind=kind
        self.embed=nn.Linear(5,WIDTH)
        self.query=nn.Linear(WIDTH,WIDTH)
        self.key=nn.Linear(WIDTH,WIDTH)
        self.value=nn.Linear(WIDTH,WIDTH)
        self.norm=nn.LayerNorm(2*WIDTH)
        self.head=nn.Sequential(nn.Linear(2*WIDTH,WIDTH),nn.GELU(),nn.Linear(WIDTH,1))
        self.memory=CCNNMemory() if kind=="ccnn" else FIRMemory() if kind=="fir" else None
        self.coupling=nn.Parameter(torch.tensor(.65)) if kind!="attention" else None
    def forward(self,x,q,k_override=None,disable_memory=False):
        t=torch.arange(x.shape[1],device=x.device,dtype=x.dtype)[None,:]/x.shape[1]
        t=t.expand_as(x)
        last=torch.zeros_like(x);last[:,-1]=1.
        token=self.embed(torch.stack((x,q,t,t.square(),last),-1))
        query=self.query(token[:,-1])
        key=self.key(token);val=self.value(token)
        scores=torch.einsum("bd,btd->bt",query,key)/math.sqrt(WIDTH)
        if self.memory is not None and not disable_memory:
            z=self.memory.trace(x,q,k_override)
            scores=scores+self.coupling*z
        scores=torch.cat((scores[:,:-1],torch.full_like(scores[:,-1:],-1e4)),1)
        att=torch.softmax(scores,-1)
        context=torch.einsum("bt,btd->bd",att,val)
        logits=self.head(self.norm(torch.cat((context,token[:,-1]),-1))).squeeze(-1)
        return logits,att

def clone_attention_parameters(source,target):
    names=("embed","query","key","value","norm","head")
    for name in names:
        getattr(target,name).load_state_dict(copy.deepcopy(getattr(source,name).state_dict()))

def attention_metrics(model,data):
    model.eval()
    with torch.no_grad():
        logits,mass=model(data["x"],data["q"])
        target=data["evidence"]
        i=torch.arange(len(target),device=DEVICE)
        base=bce(logits,data["y"])
        ablated=bce(model(data["x"],data["q"],disable_memory=True)[0],data["y"])
        return dict(accuracy=float(((logits>0).float()==data["y"]).float().mean()),
                    evidence_top1=float((mass.argmax(-1)==target).float().mean()),
                    evidence_mass=float(mass[i,target].mean()),
                    heldout_bce=float(base),
                    branch_ablation_delta_bce=float(ablated-base),
                    params=sum(p.numel() for p in model.parameters()))


In [ ]:
# 7. Training/checkpoint selection (train and validation only)
def fit_model(model,train,val,epochs):
    optimizer=torch.optim.AdamW(model.parameters(),lr=.002,weight_decay=.0001)
    best=(float("inf"),None,0);since=0;history=[];start=time.perf_counter()
    for epoch in range(epochs):
        model.train()
        ix=torch.randperm(len(train["y"]),device=DEVICE)
        for idx in ix.split(96):
            optimizer.zero_grad()
            loss=bce(get_logits(model,{"x":train["x"][idx],"q":train["q"][idx]}),train["y"][idx])
            loss.backward();optimizer.step()
        model.eval()
        with torch.no_grad():
            v=float(bce(get_logits(model,val),val["y"]))
            tr=float(bce(get_logits(model,train),train["y"]))
        history.append(dict(epoch=epoch+1,train_bce=tr,val_bce=v))
        if v<best[0]-1e-5:
            best=(v,copy.deepcopy(model.state_dict()),epoch+1);since=0
        else: since+=1
        if since>=PATIENCE:break
    model.load_state_dict(best[1]);model.eval()
    return {"best_val_bce":best[0],"selected_epoch":best[2],
            "stopped_early":len(history)<epochs,"train_seconds":time.perf_counter()-start,
            "history":history,"checkpoint_sha256":sha_tensors(model.state_dict())}

# The following loops do not use the test split for model selection.
architecture_rows=[]; models_per_seed={}; splits_per_seed={}; training_meta={}
for seed in SEEDS:
    seed_everything(seed)
    datasets=make_datasets(seed)
    splits_per_seed[seed]=datasets
    reference=AttentionMemory("attention").to(DEVICE)
    common=copy.deepcopy(reference.state_dict())
    models={}
    for kind in ("attention","ccnn","fir"):
        seed_everything(seed+{"attention":0,"ccnn":1,"fir":2}[kind])
        model=AttentionMemory(kind).to(DEVICE)
        if kind!="attention":clone_attention_parameters(reference,model)
        t=fit_model(model,datasets["train"],datasets["val"],EPOCHS)
        training_meta[f"{seed}/{kind}"]=t
        m=attention_metrics(model,datasets["test"])
        architecture_rows.append(dict(seed=seed,model=kind,**m,
                                      train_seconds=t["train_seconds"],
                                      best_epoch=t["selected_epoch"],
                                      checkpoint_sha256=t["checkpoint_sha256"]))
        models[kind]=model
    models_per_seed[seed]=models
architecture_df=pd.DataFrame(architecture_rows)
display(architecture_df.drop(columns=["checkpoint_sha256"]))
architecture_df.to_csv(OUTPUT/"attention_results.csv",index=False)


## 4. Results I — Does the CCNN branch actually help attention?

The next plots show **held-out temporal-evidence top-1**, **task accuracy**, and **branch ablation**. A positive branch-ablation delta means removing the branch worsens held-out BCE; it **does not** by itself establish improvement against a strong FIR. Attention mass and retrieval accuracy should be interpreted together.

These figures are generated only from executed model predictions. A two-epoch smoke run will commonly be undertrained and **must not** be cited as a finding.

In [ ]:
# 8. Figure 1 — attention augmentation on unseen evidence positions
fig,axes=plt.subplots(1,3,figsize=(14,3.8))
keys=["accuracy","evidence_top1","branch_ablation_delta_bce"]
titles=["Held-out retrieval accuracy ↑","Correct-event top-1 attention ↑",
        "Loss increase when branch removed ↑"]
for ax,key,title in zip(axes,keys,titles):
    y=architecture_df.groupby("model")[key].mean().reindex(["attention","ccnn","fir"])
    ax.bar(y.index,y.values)
    ax.set_title(title);ax.set_ylabel(key);ax.axhline(0,color="black",linewidth=.6)
fig.suptitle("Figure 1. Attention–CCNN vs unrestricted FIR, independent MQAR-style task")
fig.tight_layout()
fig.savefig(OUTPUT/"Figure_01_Attention_Augmentation.png",dpi=220)
plt.show()


## 5. Experimental design II — blind parameter fault and weight-space repair

We use the **same trained models** and fixed targets from Experiment I. The evaluator perturbs a chosen parameter in a copy of each model and calibrates a similar *output-level* damage on a separate calibration set. The editor is given only the faulted model and adaptation data; the fault magnitude and clean weights are not inputs to the edit algorithm.

We evaluate the following **practical minimum** set: structured CCNN edit (5 coefficients), tangent-direction CCNN edit (5 coefficients), unrestricted cosine-basis FIR correction (5 coefficients), full FIR correction (14 coefficients), plus ordinary full-model Adam continuation as an expensive optimization control. For the FIR model the same cosine/full correction and Adam controls apply. **These are controls inspired by editing studies, not exact reproductions of AlphaEdit/CrispEdit's algorithms.** More complete GN/null-space controls are required before making broad superiority claims.

Two independent tests are necessary:

\[
\mathrm{recovery}=\frac{\mathcal L_\mathrm{fault}-\mathcal L_\mathrm{repaired}}
{\mathcal L_\mathrm{fault}-\mathcal L_\mathrm{clean}},\qquad
\mathrm{specificity}=(\mathcal L_\mathrm{fault}-\mathcal L_\mathrm{repaired})-
(\mathcal L_\mathrm{clean}-\mathcal L_\mathrm{placebo}).
\]

If the fault produces negligible damage, recovery is **undefined** and must not be plotted as zero. Original/source performance is measured on a disjoint source test cohort **without any domain gate**.

In [ ]:
# 9. Fault damage calibration; zero-fault placebo; editable kernel coordinates
SOURCE_KEY=-.9
AFFECTED_KEY=.9

def group_subset(data,group):
    code=data["q"][:,-1]
    ids=(code > .65) if group=="affected" else (code < -.65)
    if int(ids.sum())<3:
        raise ValueError("Insufficient disjoint source/affected cohort")
    return {"x":data["x"][ids],"q":data["q"][ids],"y":data["y"][ids]}

def group_loss(model,data,group):
    sub=group_subset(data,group)
    model.eval()
    with torch.no_grad():
        return float(bce(get_logits(model,sub),sub["y"]))

def faulted_model(clean,magnitude):
    corrupted=copy.deepcopy(clean)
    with torch.no_grad():
        if corrupted.kind=="ccnn":
            corrupted.memory.muq.add_(magnitude)
        else:
            corrupted.memory.net[-1].bias[0].add_(magnitude)
    corrupted.eval()
    return corrupted

def calibrate_fault(clean,cal):
    before=group_loss(clean,cal,"affected")
    candidates=[.1,-.1,.25,-.25,.5,-.5,1.,-1.,2.,-2.,4.,-4.,8.,-8.]
    rows=[]
    for magnitude in candidates:
        corrupted=faulted_model(clean,magnitude)
        damage=group_loss(corrupted,cal,"affected")-before
        rows.append((damage,magnitude,corrupted))
    acceptable=[r for r in rows if r[0]>=.01]
    chosen=min(acceptable,key=lambda r:abs(r[0]-.06)) if acceptable else max(rows,key=lambda r:r[0])
    return {"model":chosen[2],"oracle_magnitude":chosen[1],
            "calibration_damage":chosen[0],"detectable":chosen[0]>=.01}

def cosine_basis(q,num=5):
    jj=torch.arange(LAGS,dtype=q.dtype,device=q.device)[:,None]
    kk=torch.arange(num,dtype=q.dtype,device=q.device)[None,:]
    return F.normalize(torch.cos(torch.pi*(jj+.5)*kk/LAGS),dim=0,eps=1e-7)

class EditKernel(nn.Module):
    def __init__(self,faulted,method):
        super().__init__()
        self.base=copy.deepcopy(faulted)
        for p in self.base.parameters():p.requires_grad_(False)
        self.method=method
        budget=14 if method=="full14" else 5
        self.delta=nn.Parameter(torch.zeros(budget,device=DEVICE))
    def kernel(self,q):
        branch=self.base.memory
        if self.method=="structured5":
            assert self.base.kind=="ccnn"
            return branch.weights(q,delta=self.delta)
        if self.method=="tangent5":
            assert self.base.kind=="ccnn"
            p=branch.params(); base=branch.weights(q)
            update=torch.zeros_like(base)
            for i in range(5):
                plus=torch.zeros_like(self.delta);plus[i]=.001
                minus=-plus
                direction=(branch.weights(q,plus)-branch.weights(q,minus))/.002
                update=update+self.delta[i]*direction
            return F.normalize(base+update,dim=-1,eps=1e-7)
        base=branch.weights(q)
        if self.method=="spectral5":
            offset=cosine_basis(q,5)@self.delta
        elif self.method=="full14":
            offset=self.delta
        else: raise ValueError(self.method)
        return F.normalize(base+offset.view(1,1,-1),dim=-1,eps=1e-7)
    def forward(self,x,q):
        return self.base(x,q,k_override=self.kernel(q))[0]

def fit_edit(faulted,train,val,source_ceiling,method,steps):
    # Eligible source/affected labels remain fixed and include no test examples.
    if method=="continued_adam":
        adapted=copy.deepcopy(faulted)
        opt=torch.optim.Adam(adapted.parameters(),lr=.001)
        count=sum(p.numel() for p in adapted.parameters() if p.requires_grad)
    else:
        adapted=EditKernel(faulted,method)
        opt=torch.optim.Adam([adapted.delta],lr=.02)
        count=adapted.delta.numel()
    def forward(data):
        return adapted(data["x"],data["q"])[0] if method=="continued_adam" else adapted(data["x"],data["q"])
    def val_score():
        adapted.eval()
        with torch.no_grad():
            src=group_subset(val,"source");aff=group_subset(val,"affected")
            return float(bce(forward(src),src["y"])),float(bce(forward(aff),aff["y"]))
    baseline_source,baseline_affected=val_score()
    # Candidate acceptance and checkpoint selection use validation *only*.
    best=None
    for step in range(steps):
        adapted.train();opt.zero_grad()
        aff=group_subset(train,"affected");src=group_subset(train,"source")
        loss=bce(forward(aff),aff["y"])+bce(forward(src),src["y"])
        loss.backward();opt.step()
        src_val,aff_val=val_score()
        if src_val <= source_ceiling and aff_val < baseline_affected*.995:
            if best is None or aff_val<best[0]:
                best=(aff_val,copy.deepcopy(adapted.state_dict()),step+1)
    if best is None:
        return faulted,False,0,count,"validation_rejected"
    adapted.load_state_dict(best[1]);adapted.eval()
    if method=="continued_adam":
        return adapted,True,best[2],count,"validation_accepted"
    return adapted,True,best[2],count,"validation_accepted"

def model_logit(model,data):
    if isinstance(model,EditKernel):
        return model(data["x"],data["q"])
    return get_logits(model,data)
def loss_cohort(model,data,group):
    sub=group_subset(data,group)
    model.eval()
    with torch.no_grad():return float(bce(model_logit(model,sub),sub["y"]))


In [ ]:
# 10. Paired repair study: same split and checkpoint, separate placebo and fault
repair_rows=[];calibration_rows=[]
for seed in SEEDS:
    datasets=splits_per_seed[seed]
    for kind in ("ccnn","fir"):
        clean=models_per_seed[seed][kind]
        calibration=calibrate_fault(clean,datasets["cal"])
        fault=calibration["model"]
        calibration_rows.append(dict(seed=seed,model=kind,
                                     calibrated_damage=calibration["calibration_damage"],
                                     fault_detectable_on_calibration=calibration["detectable"],
                                     evaluator_only_magnitude=calibration["oracle_magnitude"]))
        clean_val_source=loss_cohort(clean,datasets["val"],"source")
        # Validation preservation is relative to clean baseline, never test.
        ceiling=clean_val_source + .03
        methods=(["structured5","tangent5","spectral5","full14","continued_adam"]
                 if kind=="ccnn" else
                 ["spectral5","full14","continued_adam"])
        for method in methods:
            clock=time.perf_counter()
            repaired,accepted,step_count,parameters,status=fit_edit(
                fault,datasets["train"],datasets["val"],ceiling,method,REPAIR_STEPS)
            sham,sham_accepted,_,_,_=fit_edit(
                clean,datasets["train"],datasets["val"],ceiling,method,REPAIR_STEPS)
            affected_before=loss_cohort(fault,datasets["test"],"affected")
            affected_clean=loss_cohort(clean,datasets["test"],"affected")
            affected_after=loss_cohort(repaired,datasets["test"],"affected")
            affected_placebo=loss_cohort(sham,datasets["test"],"affected")
            source_clean=loss_cohort(clean,datasets["test"],"source")
            source_after=loss_cohort(repaired,datasets["test"],"source")
            headroom=affected_before-affected_clean
            recover=affected_before-affected_after
            placebo=affected_clean-affected_placebo
            supported=(calibration["detectable"] and headroom > .01)
            repair_rows.append(dict(seed=seed,model=kind,method=method,
                calibrated_fault_detectable=calibration["detectable"],
                test_fault_headroom=headroom,
                recovered_fraction=(recover/headroom if supported else np.nan),
                test_fault_improvement=recover,
                test_placebo_improvement=placebo,
                specificity_excess=recover-placebo,
                test_source_clean=source_clean,
                test_source_after=source_after,
                source_drift=source_after-source_clean,
                source_preserved=source_after<=source_clean+.03,
                accepted_on_validation=accepted,
                placebo_accepted_on_validation=sham_accepted,
                best_step=step_count,editable_parameters=parameters,
                elapsed_seconds=time.perf_counter()-clock,
                status=("undetectable_fault" if not supported else status)))
repair_df=pd.DataFrame(repair_rows)
calibration_df=pd.DataFrame(calibration_rows)
display(calibration_df)
display(repair_df.drop(columns=["elapsed_seconds"]).round(4))
repair_df.to_csv(OUTPUT/"repair_results.csv",index=False)
calibration_df.to_csv(OUTPUT/"fault_calibration.csv",index=False)


## 6. Results II — fault recovery versus source preservation

Interpret Figure 2 as a **trade-off**, not as a rank ordering. A method counts as a fault-specific candidate only when its calibrated fault is detectable, the repair was accepted based on validation, the untouched source test stays within the predeclared tolerance, and the held-out improvement exceeds the no-fault placebo.

**No cherry-picking:** all rejected edits, tiny-fault cases, source regressions and neutral/negative findings remain in the CSV.

In [ ]:
# 11. Figure 2 — fault-specific editing vs unintended drift
fig, ax=plt.subplots(figsize=(8.5,5))
for model_name,group in repair_df.groupby("model"):
    ax.scatter(group["source_drift"],group["specificity_excess"],label=model_name,s=52,alpha=.85)
    for _,r in group.iterrows():
        ax.annotate(r["method"],(r["source_drift"],r["specificity_excess"]),
                    fontsize=7,xytext=(4,3),textcoords="offset points")
ax.axvline(.03,linestyle="--",color="gray",label="Source tolerance")
ax.axhline(0,linestyle=":",color="black")
ax.set_xlabel("Source test BCE drift vs clean (lower is better)")
ax.set_ylabel("Fault-specific BCE gain over placebo (higher is better)")
ax.set_title("Figure 2. Recovery–preservation trade-off, fixed-label model faults")
if not repair_df["calibrated_fault_detectable"].any():
    ax.text(.5,.5,"NO DETECTABLE FAULT: NO REPAIR INFERENCE",
            transform=ax.transAxes,ha="center",va="center",fontsize=10,
            bbox=dict(facecolor="white",edgecolor="gray",alpha=.9))
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT/"Figure_02_Repair_Tradeoff.png",dpi=220)
plt.show()


In [ ]:
# 12. Audit manifest and downloadable numerical evidence
manifest={
    "research_question":"Does original CCNN lag structure enhance attention and weight-space repair?",
    "evidence_status":"engineering_smoke" if FAST_MODE else "exploratory_pilot",
    "confirmatory":False,
    "paper_method_basis":["Zoology (ICLR 2024) MQAR","Samba (ICLR 2025) passkey/phonebook",
        "Hymba (ICLR 2025) hybrid ablations","AlphaEdit (ICLR 2025) locality",
        "CrispEdit (ICML 2026) constrained editing","Mirage (ACL 2025) realistic evaluation"],
    "task":"Adapted MQAR-style cue–key retrieval with unseen event positions",
    "canonical_mqar_reproduced":False,
    "canonical_mqar_generator":"https://github.com/HazyResearch/zoology/blob/main/zoology/data/multiquery_ar.py",
    "all_target_labels_from_independent_task":True,
    "fault_intervention":"model weight changes; no teacher/label shift",
    "fault_oracle_hidden_from_editor":"procedural only, not secure isolation",
    "notebook_architecture":"direct CCNN/PDE-ODE trace -> attention logits, matched FIR alternative",
    "seeds":SEEDS,"epochs_cap":EPOCHS,"repair_steps":REPAIR_STEPS,
    "split_sizes":{"train":TRAIN_N,"validation":VAL_N,"calibration":CAL_N,"test":TEST_N},
    "device":str(DEVICE),"torch_version":torch.__version__,
    "source_commit":"fill_with_git_commit_for_colab_run",
    "training_checkpoints":{k:{"sha256":v["checkpoint_sha256"],
                               "selected_epoch":v["selected_epoch"],
                               "val_bce":v["best_val_bce"],
                               "training_seconds":v["train_seconds"]} 
                            for k,v in training_meta.items()},
    "limitations":["Not the original published MQAR benchmark",
       "Small 3-arm pilot lacks full SSM/Neural-ODE/modern convolution baseline",
       "Training/FLOPs/hyperparameter parity not yet independently established",
       "Fault calibration may be undetectable on underused memory module",
       "Procedural blindness; evaluator/model not securely separated",
       "No multi-seed confirmatory result or independent replication"]
}
(OUTPUT/"experiment_manifest.json").write_text(json.dumps(manifest,indent=2))
display(pd.DataFrame([{"artifact":p.name,"bytes":p.stat().st_size}
 for p in sorted(OUTPUT.iterdir()) if p.is_file()]))
print("Evidence is saved in:",OUTPUT.resolve(),
      "\nNOT publication-ready:",manifest["evidence_status"])


## 7. Discussion (complete after execution, without inventing positive results)

**Decision table** — fill from the exported CSVs and locked evaluation:

| Component | Supporting evidence required | Evidence currently established |
|---|---|---|
| CCNN attention augmentation | Higher held-out key retrieval/temporal selectivity than tuned FIR and attention; meaningful memory-ablation effect | **Not established before running this notebook** |
| Mechanistic weight repair | Detectable weight fault; positive *placebo-adjusted* recovery; independent source preservation; better recovery/cost than tangent/FIR/optimizer controls | **Not established before running this notebook** |
| Full joint hypothesis | Both effects on the same task, stable across paired independent seeds and compute | **Not established before confirmatory study** |

**Interpretation caveats:** An attention-logit bias generated from the original CCNN lag integral and leaky dynamics is a *proposed coupling*. It is not equivalent to claiming that the 2021 paper itself tested attention. An output ablation alone does not prove causal reliance on a particular physical mechanism. Functional damage matching across distinct parameterizations remains a confound; calibration on a held-out subset mitigates but does not eliminate this. Task structure may still favor or disfavor certain priors. Models must be sufficiently trained and fairly tuned.

> The preliminary experiments demonstrate the feasibility of structure-aware convolutional repair, but the observed advantage is conditional on the data-generating mechanism and the choice of repair parameterization. The present evidence is insufficient to establish superiority over expressive unstructured filters or conventional optimization methods.

### Next acceptance gates

The larger exploratory run is **not** confirmation. Before a paper claim: freeze the metric and hyperparameter protocol, include a strong SSM/generic Neural ODE as required, measure actual FLOPs/compute, run **at least 10 paired independent seeds**, compare editing effects after calibrated matched functional damage, secure evaluator-only fault metadata, and independently replicate. Keep all failed/no-fault results.

## 8. References and reproducibility sources

1. Habiba & Pearlmutter (2021). [*Continuous Convolutional Neural Networks: Coupled Neural PDE and ODE*](https://arxiv.org/abs/2111.00343). Defining lag-integral mechanism.
2. Arora et al. (ICLR 2024). [*Zoology: Measuring and Improving Recall in Efficient Language Models*](https://proceedings.iclr.cc/paper_files/paper/2024/hash/448fc91f669c15d10364ee01d512cc10-Abstract-Conference.html). MQAR and convolution–attention hybrid recall.
3. Ren et al. (ICLR 2025). [*Samba: Simple Hybrid State Space Models for Efficient Unlimited Context Language Modeling*](https://proceedings.iclr.cc/paper_files/paper/2025/hash/84a7fc24ed52e8eff514c33e8ac76ea3-Abstract-Conference.html). Passkey, phonebook extrapolation.
4. Dong et al. (ICLR 2025). [*Hymba: A Hybrid-head Architecture for Small Language Models*](https://proceedings.iclr.cc/paper_files/paper/2025/hash/f32def07618040e540e0a6182e290562-Abstract-Conference.html). Parallel memory heads and hybrid ablations.
5. Fang et al. (ICLR 2025). [*AlphaEdit: Null-Space Constrained Knowledge Editing for Language Models*](https://proceedings.iclr.cc/paper_files/paper/2025/hash/29c8c615b3187ee995029284702d3f43-Abstract-Conference.html). Preservation and sequential edits.
6. Ikram et al. (ICML 2026). [*CrispEdit: Low-Curvature Projections for Scalable Non-Destructive LLM Editing*](https://proceedings.mlr.press/v306/ikram26a.html). Constrained editing and retention.
7. Yang et al. (ACL 2025). [*The Mirage of Model Editing: Revisiting Evaluation in the Wild*](https://aclanthology.org/2025.acl-long.745/). Evaluation leakage/teacher forcing.

**Governing research charter:** [RESEARCH_QUESTIONS_CCNN_ATTENTION_WEIGHT_REPAIR.md](../notebooks/archive/RESEARCH_QUESTIONS_CCNN_ATTENTION_WEIGHT_REPAIR.md).

**Caution:** This notebook is an auditable starting point, not an experiment certified as scientifically valid by the authors of the referenced papers.